# Start here: an independently checked population calculation

The complete demo is runnable from the README. This short notebook focuses on what you must understand before scaling up. It uses no public GW data and no cluster. The numerical examples are not a validated astrophysical model or a real-data conclusion.

Before running a cell, predict the result. Student A should explain the event likelihood; Student B should explain population normalization and selection. Both should explain why later events share a common population parameter.


In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src" / "gwtc_forecast").exists():
    root = root.parent
if not (root / "src" / "gwtc_forecast").exists():
    raise RuntimeError("Launch from the repository or its notebooks directory")
sys.path.insert(0, str(root / "src"))
import numpy as np
from scipy.integrate import quad
from gwtc_forecast.experiments.feature_demo import load_demo_config, build_pair
config, _ = load_demo_config(root / "config" / "feature_demo.toml")
models, priors = build_pair(config)
parameters = config["generating_parameters"]


## 1. Does the population integrate to one?
The density is defined per unit mass, not per unit log mass. Compare an independent adaptive integral with the model normalization.

In [2]:
for model in models:
    value, error = quad(
        lambda mass: float(np.exp(model.log_prob(
            {"mass_1_source": np.array([mass])}, parameters))[0]),
        model.m_min, model.m_max, epsabs=1e-9,
    )
    print(model.name, "integral:", value, "quadrature error:", error)
    assert abs(value - 1) < 1e-6


continuum_lowpeak integral: 1.0000000000000246 quadrature error: 6.94799216598152e-09
continuum_lowpeak_highpeak integral: 1.0000000000000215 quadrature error: 7.363080089949271e-09


## 2. Do the models really differ by one added component?
Set its mixture weight to zero and predict the relationship. This is a structural check, not evidence for or against that component.

In [3]:
mass = np.geomspace(5, 160, 1000)
without_high = {**parameters, "f_high": 0.0}
a = models[0].log_prob({"mass_1_source": mass}, without_high)
b = models[1].log_prob({"mass_1_source": mass}, without_high)
print("Maximum difference at zero high-component weight:", np.max(np.abs(a-b)))
assert np.allclose(a, b, atol=1e-12)


Maximum difference at zero high-component weight: 0.0


## 3. Why do independent event averages not equal a joint prediction?
The following numbers are illustrative likelihood factors. Perform the two sums by hand before running the cell.

In [4]:
weights = np.array([0.4, 0.6])
factors = np.array([[0.2, 0.8], [0.7, 0.1]])
joint = np.log(weights @ np.prod(factors, axis=0))
independent = np.sum(np.log(factors @ weights))
print("Joint log score:", joint, "Sum of independent log scores:", independent)
assert not np.isclose(joint, independent)


Joint log score: -2.2633643798407643 Sum of independent log scores: -1.6586281566248724


## What to do next

Run the README `feature-demo` command in a new external or `/tmp` output directory. Read its report and all limitations. Do not infer that a positive demonstration score predicts an astronomical result: its generating population contains an excess by construction.

For your first independent contribution, build and explain a known-answer event-reweighting or selection calculation that is not simply a rerun of these supplied assertions. Use `STUDENT_MEETING_MILESTONES.md` and bring `docs/MEETING_REPORT_TEMPLATE.md` to the meeting.